In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams['font.family'] ='Malgun Gothic'
plt.rcParams['axes.unicode_minus'] =False

In [ ]:
import pandas as pd

dfa = pd.read_csv('rtu_module_12(4호기).csv')
dfb = pd.read_csv('rtu_module_13(3호기).csv')
dfc = pd.read_csv('rtu_module_14(2호기).csv')
dfd = pd.read_csv('rtu_module_17(6호기).csv')

df2 = dfa.copy()
df3 = dfb.copy()
df4 = dfc.copy()
df6 = dfd.copy()

## 시계열 데이터 변환

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def initial_preprocess(df):
    # 1. 시계열 변환: 20241201000000 -> datetime
    df['localtime'] = pd.to_datetime(df['localtime'].astype(str), format='%Y%m%d%H%M%S')
    
    # 2. 시간 분해 변수
    df['date'] = df['localtime'].dt.date
    df['month'] = df['localtime'].dt.month
    df['hour'] = df['localtime'].dt.hour
    df['weekday'] = df['localtime'].dt.weekday # 0:월, 6:일
    
    # 3. 에너지 소비량(차분) 및 평균 역률 계산
    df['energy_diff'] = df['accumActiveEnergy'].diff().fillna(0)
    df['pf_avg'] = df[['powerFactorR', 'powerFactorS', 'powerFactorT']].mean(axis=1)
    
    # 4. 공휴일 정의
    holidays = ['2024-12-25', '2025-01-01', '2025-01-27', '2025-01-28', 
                '2025-01-29', '2025-01-30', '2025-03-01', '2025-03-03']
    df['is_holiday'] = df['localtime'].dt.date.astype(str).isin(holidays)
    df['is_off_day'] = df['is_holiday'] | (df['weekday'] >= 5)
    
    return df

df_list = [df2, df3, df4, df6]
names = ['Unit 2', 'Unit 3', 'Unit 4', 'Unit 6']
for i in range(len(df_list)):
    df_list[i] = initial_preprocess(df_list[i])

# EDA

## 평일 vs 주말 및 공휴일 패턴

In [ ]:
for df, name in zip(df_list, names):
    work_vs_off = df.groupby(['is_off_day', 'hour'])['activePower'].mean().unstack(level=0)
    
    work_peak_hr = work_vs_off[False].idxmax()
    off_peak_hr = work_vs_off[True].idxmax()
    
    print(f"[{name}] 평일 피크: {work_peak_hr}시, 휴일 피크: {off_peak_hr}시 (차이: {abs(work_peak_hr - off_peak_hr)}시간)")

### 유효전력(ActivePower)

In [ ]:
def plot_work_vs_off_active(df, name):
    plt.figure(figsize=(12, 4))
    
    # 평일 vs 휴일 시간대별 평균 비교
    work_vs_off = df.groupby(['is_off_day', 'hour'])['activePower'].mean().unstack(level=0)
    
    plt.subplot(1, 2, 1)
    plt.plot(work_vs_off[False], label='Work Day')
    plt.plot(work_vs_off[True], label='Weekend/Holiday', linestyle='--')
    plt.title(f"{name}: Work vs Off-day")
    plt.legend()

    # 월별 전체적인 전력 사용 추이 (Boxplot)
    plt.subplot(1, 2, 2)
    sns.boxplot(x='month', y='activePower', data=df)
    plt.title(f"{name}: Monthly Distribution")
    
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_work_vs_off_active(df, name)

#### + 월별로 나눠서 보기

In [ ]:
def plot_monthly_work_vs_off_comparison_active(df, name):
    pivot_df = df.pivot_table(index='hour', columns=['month', 'is_off_day'], 
                              values='activePower', aggfunc='mean')
    
    months = sorted(df['month'].unique())
    n_months = len(months)
    
    fig, axes = plt.subplots(1, n_months, figsize=(5 * n_months, 5), sharey=True)
    
    if n_months == 1: axes = [axes]
    
    for i, month in enumerate(months):
        try:
            axes[i].plot(pivot_df[month][False], label='Work Day', color='royalblue', linewidth=2)
        except KeyError: pass
            
        try:
            axes[i].plot(pivot_df[month][True], label='Off Day', color='orange', linestyle='--', linewidth=2)
        except KeyError: pass
        
        axes[i].set_title(f"Month {month}", fontsize=13, fontweight='bold')
        axes[i].set_xlabel("Hour")
        axes[i].grid(True, alpha=0.3)
        if i == 0:
            axes[i].set_ylabel("Average Active Power (W)")
            axes[i].legend()

    plt.suptitle(f"[{name}] Monthly Work vs Off-day Hourly Pattern", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_monthly_work_vs_off_comparison_active(df, name)

### 지연 무효전력(ReactivePowerLagging)

In [ ]:
def plot_work_vs_off_reactive(df, name):
    plt.figure(figsize=(12, 4))
    
    # 평일 vs 휴일 시간대별 평균 비교
    work_vs_off = df.groupby(['is_off_day', 'hour'])['reactivePowerLagging'].mean().unstack(level=0)
    
    plt.subplot(1, 2, 1)
    plt.plot(work_vs_off[False], label='Work Day')
    plt.plot(work_vs_off[True], label='Weekend/Holiday', linestyle='--')
    plt.title(f"{name}: Work vs Off-day")
    plt.legend()

    # 월별 전체적인 전력 사용 추이 (Boxplot)
    plt.subplot(1, 2, 2)
    sns.boxplot(x='month', y='reactivePowerLagging', data=df)
    plt.title(f"{name}: Monthly Distribution")
    
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_work_vs_off_reactive(df, name)

#### + 월별로 나눠서 보기

In [ ]:
def plot_monthly_work_vs_off_comparison_reactive(df, name):
    pivot_df = df.pivot_table(index='hour', columns=['month', 'is_off_day'], 
                              values='reactivePowerLagging', aggfunc='mean')
    
    months = sorted(df['month'].unique())
    n_months = len(months)
    
    fig, axes = plt.subplots(1, n_months, figsize=(5 * n_months, 5), sharey=True)
    
    if n_months == 1: axes = [axes]
    
    for i, month in enumerate(months):
        try:
            axes[i].plot(pivot_df[month][False], label='Work Day', color='royalblue', linewidth=2)
        except KeyError: pass
            
        try:
            axes[i].plot(pivot_df[month][True], label='Off Day', color='orange', linestyle='--', linewidth=2)
        except KeyError: pass
        
        axes[i].set_title(f"Month {month}", fontsize=13, fontweight='bold')
        axes[i].set_xlabel("Hour")
        axes[i].grid(True, alpha=0.3)
        if i == 0:
            axes[i].set_ylabel("Average Reactive Power")
            axes[i].legend()

    plt.suptitle(f"[{name}] Monthly Work vs Off-day Hourly Pattern", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_monthly_work_vs_off_comparison_reactive(df, name)

### 누적 소비량(accumActiveEnergy)

In [ ]:
def plot_work_vs_off_acc(df, name):
    plt.figure(figsize=(12, 4))
    
    # 평일 vs 휴일 시간대별 평균 비교
    work_vs_off = df.groupby(['is_off_day', 'hour'])['accumActiveEnergy'].mean().unstack(level=0)
    
    plt.subplot(1, 2, 1)
    plt.plot(work_vs_off[False], label='Work Day')
    plt.plot(work_vs_off[True], label='Weekend/Holiday', linestyle='--')
    plt.title(f"{name}: Work vs Off-day")
    plt.legend()

    # 월별 전체적인 전력 사용 추이 (Boxplot)
    plt.subplot(1, 2, 2)
    sns.boxplot(x='month', y='accumActiveEnergy', data=df)
    plt.title(f"{name}: Monthly Distribution")
    
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_work_vs_off_acc(df, name)

#### + 월별로 나눠서 보기

In [ ]:
def plot_monthly_work_vs_off_comparison_acc(df, name):
    pivot_df = df.pivot_table(index='hour', columns=['month', 'is_off_day'], 
                              values='accumActiveEnergy', aggfunc='mean')
    
    months = sorted(df['month'].unique())
    n_months = len(months)
    
    fig, axes = plt.subplots(1, n_months, figsize=(5 * n_months, 5), sharey=True)
    
    if n_months == 1: axes = [axes]
    
    for i, month in enumerate(months):
        try:
            axes[i].plot(pivot_df[month][False], label='Work Day', color='royalblue', linewidth=2)
        except KeyError: pass
            
        try:
            axes[i].plot(pivot_df[month][True], label='Off Day', color='orange', linestyle='--', linewidth=2)
        except KeyError: pass
        
        axes[i].set_title(f"Month {month}", fontsize=13, fontweight='bold')
        axes[i].set_xlabel("Hour")
        axes[i].grid(True, alpha=0.3)
        if i == 0:
            axes[i].set_ylabel("Average Accumulate Energy")
            axes[i].legend()

    plt.suptitle(f"[{name}] Monthly Work vs Off-day Hourly Pattern", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_monthly_work_vs_off_comparison_acc(df, name)

### 에너지 소비량(energy_diff)

In [ ]:
def plot_work_vs_off_diff(df, name):
    plt.figure(figsize=(12, 4))
    
    # 평일 vs 휴일 시간대별 평균 비교
    work_vs_off = df.groupby(['is_off_day', 'hour'])['energy_diff'].mean().unstack(level=0)
    
    plt.subplot(1, 2, 1)
    plt.plot(work_vs_off[False], label='Work Day')
    plt.plot(work_vs_off[True], label='Weekend/Holiday', linestyle='--')
    plt.title(f"{name}: Work vs Off-day")
    plt.legend()

    # 월별 전체적인 전력 사용 추이 (Boxplot)
    plt.subplot(1, 2, 2)
    sns.boxplot(x='month', y='energy_diff', data=df)
    plt.title(f"{name}: Monthly Distribution")
    
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_work_vs_off_diff(df, name)

#### + 월별로 나눠서 보기

In [ ]:
def plot_monthly_work_vs_off_comparison_diff(df, name):
    pivot_df = df.pivot_table(index='hour', columns=['month', 'is_off_day'], 
                              values='energy_diff', aggfunc='mean')
    
    months = sorted(df['month'].unique())
    n_months = len(months)
    
    fig, axes = plt.subplots(1, n_months, figsize=(5 * n_months, 5), sharey=True)
    
    if n_months == 1: axes = [axes]
    
    for i, month in enumerate(months):
        try:
            axes[i].plot(pivot_df[month][False], label='Work Day', color='royalblue', linewidth=2)
        except KeyError: pass
            
        try:
            axes[i].plot(pivot_df[month][True], label='Off Day', color='orange', linestyle='--', linewidth=2)
        except KeyError: pass
        
        axes[i].set_title(f"Month {month}", fontsize=13, fontweight='bold')
        axes[i].set_xlabel("Hour")
        axes[i].grid(True, alpha=0.3)
        if i == 0:
            axes[i].set_ylabel("Average Energy Difference")
            axes[i].legend()

    plt.suptitle(f"[{name}] Monthly Work vs Off-day Hourly Pattern", fontsize=16, y=1.05)
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_monthly_work_vs_off_comparison_diff(df, name)

## 요일별 패턴

### 유효전력(ActivePower)

In [ ]:
def plot_monthly_weekday_pattern_active(df, name):
    # 1. 요일별/월별 activePower 평균 계산
    # unstack()을 사용하여 월(month)을 컬럼으로 보냅니다.
    monthly_weekday = df.groupby(['month', 'weekday'])['activePower'].mean().unstack(level=0)
    
    plt.figure(figsize=(12, 6))
    
    # 2. 모든 달을 순회하며 선 그리기
    for month in monthly_weekday.columns:
        plt.plot(monthly_weekday.index, monthly_weekday[month], 
                 marker='o', label=f'Month {month}')
    
    # 3. 시각적 디테일 추가
    plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    plt.title(f"{name}: Weekday Pattern Comparison by Month", fontsize=14)
    plt.xlabel("Day of the Week")
    plt.ylabel("Average Active Power")
    plt.legend(title="Month", bbox_to_anchor=(1.05, 1), loc='upper left') # 범례를 바깥으로 빼서 가독성 확보
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# 실행
for df, name in zip(df_list, names):
    plot_monthly_weekday_pattern_active(df, name)

#### 전체 vs 12월

In [ ]:
def plot_weekday_pattern_active(df, name, n):
    # 전체 평균
    entire_mean = df.groupby('weekday')['activePower'].mean()
    # 월별 평균
    month_mean = df[df['month'] == n].groupby('weekday')['activePower'].mean()
    
    plt.figure(figsize=(10, 4))
    plt.plot(entire_mean.index, entire_mean.values, marker='o', label='Entire Avg')
    plt.plot(month_mean.index, month_mean.values, marker='s', linestyle='--', label=f'Month {n} Avg')
    plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    plt.title(f"{name}: Weekday Pattern Comparison")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

for df, name in zip(df_list, names):
    plot_weekday_pattern_active(df, name, 12)

#### 전체 vs 1월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_active(df, name, 1)

#### 전체 vs 2월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_active(df, name, 2)

#### 전체 vs 3월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_active(df, name, 3)

#### 전체 vs 4월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_active(df, name, 4)

### 지연 무효전력(ReactivePowerLagging)

In [ ]:
def plot_monthly_weekday_pattern_reactive(df, name):
    # 1. 요일별/월별 activePower 평균 계산
    monthly_weekday = df.groupby(['month', 'weekday'])['reactivePowerLagging'].mean().unstack(level=0)
    
    plt.figure(figsize=(12, 6))
    
    # 2. 모든 달을 순회하며 선 그리기
    for month in monthly_weekday.columns:
        plt.plot(monthly_weekday.index, monthly_weekday[month], 
                 marker='o', label=f'Month {month}')
    
    # 3. 시각적 디테일 추가
    plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    plt.title(f"{name}: Weekday Pattern Comparison by Month", fontsize=14)
    plt.xlabel("Day of the Week")
    plt.ylabel("Average Reactive Power")
    plt.legend(title="Month", bbox_to_anchor=(1.05, 1), loc='upper left') # 범례를 바깥으로 빼서 가독성 확보
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# 실행
for df, name in zip(df_list, names):
    plot_monthly_weekday_pattern_reactive(df, name)

#### 전체 vs 12월

In [ ]:
def plot_weekday_pattern_reactive(df, name, n):
    # 전체 평균
    entire_mean = df.groupby('weekday')['reactivePowerLagging'].mean()
    # 월별 평균
    month_mean = df[df['month'] == n].groupby('weekday')['reactivePowerLagging'].mean()
    
    plt.figure(figsize=(10, 4))
    plt.plot(entire_mean.index, entire_mean.values, marker='o', label='Entire Avg')
    plt.plot(month_mean.index, month_mean.values, marker='s', linestyle='--', label=f'Month {n} Avg')
    plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    plt.title(f"{name}: Weekday Pattern Comparison")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

for df, name in zip(df_list, names):
    plot_weekday_pattern_reactive(df, name, 12)

#### 전체 vs 1월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_reactive(df, name, 1)

#### 전체 vs 2월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_reactive(df, name, 2)

#### 전체 vs 3월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_reactive(df, name, 3)

#### 전체 vs 4월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_reactive(df, name, 4)

### 누적 소비량(accumActiveEnergy)

In [ ]:
def plot_monthly_weekday_pattern_acc(df, name):
    # 1. 요일별/월별 activePower 평균 계산
    monthly_weekday = df.groupby(['month', 'weekday'])['accumActiveEnergy'].mean().unstack(level=0)
    
    plt.figure(figsize=(12, 6))
    
    # 2. 모든 달을 순회하며 선 그리기
    for month in monthly_weekday.columns:
        plt.plot(monthly_weekday.index, monthly_weekday[month], 
                 marker='o', label=f'Month {month}')
    
    # 3. 시각적 디테일 추가
    plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    plt.title(f"{name}: Weekday Pattern Comparison by Month", fontsize=14)
    plt.xlabel("Day of the Week")
    plt.ylabel("Average Accumulate Energy")
    plt.legend(title="Month", bbox_to_anchor=(1.05, 1), loc='upper left') # 범례를 바깥으로 빼서 가독성 확보
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# 실행
for df, name in zip(df_list, names):
    plot_monthly_weekday_pattern_acc(df, name)

In [ ]:
def plot_weekday_pattern_acc(df, name, n):
    # 전체 평균
    entire_mean = df.groupby('weekday')['accumActiveEnergy'].mean()
    # 월별 평균
    month_mean = df[df['month'] == n].groupby('weekday')['accumActiveEnergy'].mean()
    
    plt.figure(figsize=(10, 4))
    plt.plot(entire_mean.index, entire_mean.values, marker='o', label='Entire Avg')
    plt.plot(month_mean.index, month_mean.values, marker='s', linestyle='--', label=f'Month {n} Avg')
    plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    plt.title(f"{name}: Weekday Pattern Comparison")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

for df, name in zip(df_list, names):
    plot_weekday_pattern_acc(df, name, 12)

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_acc(df, name, 1)

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_reactive(df, name, 2)

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_reactive(df, name, 3)

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_reactive(df, name, 4)

### 에너지 소비량(energy_diff)

In [ ]:
def plot_monthly_weekday_pattern_diff(df, name):
    # 1. 요일별/월별 activePower 평균 계산
    monthly_weekday = df.groupby(['month', 'weekday'])['energy_diff'].mean().unstack(level=0)
    
    plt.figure(figsize=(12, 6))
    
    # 2. 모든 달을 순회하며 선 그리기
    for month in monthly_weekday.columns:
        plt.plot(monthly_weekday.index, monthly_weekday[month], 
                 marker='o', label=f'Month {month}')
    
    # 3. 시각적 디테일 추가
    plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    plt.title(f"{name}: Weekday Pattern Comparison by Month", fontsize=14)
    plt.xlabel("Day of the Week")
    plt.ylabel("Average Energy Difference")
    plt.legend(title="Month", bbox_to_anchor=(1.05, 1), loc='upper left') # 범례를 바깥으로 빼서 가독성 확보
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

# 실행
for df, name in zip(df_list, names):
    plot_monthly_weekday_pattern_diff(df, name)

#### 전체 vs 12월

In [ ]:
def plot_weekday_pattern_diff(df, name, n):
    # 전체 평균
    entire_mean = df.groupby('weekday')['energy_diff'].mean()
    # 월별 평균
    month_mean = df[df['month'] == n].groupby('weekday')['energy_diff'].mean()
    
    plt.figure(figsize=(10, 4))
    plt.plot(entire_mean.index, entire_mean.values, marker='o', label='Entire Avg')
    plt.plot(month_mean.index, month_mean.values, marker='s', linestyle='--', label=f'Month {n} Avg')
    plt.xticks(range(7), ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
    plt.title(f"{name}: Weekday Pattern Comparison")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

for df, name in zip(df_list, names):
    plot_weekday_pattern_diff(df, name, 12)

#### 전체 vs 1월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_diff(df, name, 1)

#### 전체 vs 2월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_diff(df, name, 2)

#### 전체 vs 3월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_diff(df, name, 3)

#### 전체 vs 4월

In [ ]:
for df, name in zip(df_list, names):
    plot_weekday_pattern_diff(df, name, 4)

## 시간대별 패턴

### 유효전력(ActivePower)

In [ ]:
def plot_hourly_pattern_active(df, name):
    
    monthly_hourly = df.pivot_table(index='hour', columns='month', values='activePower', aggfunc='mean')
    overall_hourly = df.groupby('hour')['activePower'].mean()
    
    plt.figure(figsize=(12, 6))
    
    colors = sns.color_palette("husl", len(monthly_hourly.columns))
    
    for i, month in enumerate(monthly_hourly.columns):
        plt.plot(monthly_hourly.index, monthly_hourly[month], 
                 label=f'Month {month}', color=colors[i], marker='.', alpha=0.8)
    
    plt.plot(overall_hourly.index, overall_hourly.values, 
             color='black', linewidth=2, label='OVERALL AVG', linestyle='--')
    
    plt.title(f"{name}: Hourly Active Power Trend by Month", fontsize=15)
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Average Active Power")
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', alpha=0.3)
    plt.legend(title="Months", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

# 실행
for df, name in zip(df_list, names):
    plot_hourly_pattern_active(df, name)

#### 전체 vs 12월

In [ ]:
def plot_hourly_comparison_active(df, name, n):

    overall_hourly = df.groupby('hour')['activePower'].mean()
    month_hourly = df[df['month'] == n].groupby('hour')['activePower'].mean()
    
    plt.figure(figsize=(12, 5))
  
    plt.plot(overall_hourly.index, overall_hourly.values, 
             color='lightgray', linewidth=3, label='Entire Avg', marker='o', alpha=0.6)
    plt.plot(month_hourly.index, month_hourly.values, 
             color='darkorange', linewidth=2, label=f'Month {n} Avg', marker='s', linestyle='--')
    plt.title(f"{name}: Hourly Reactive Power (Overall vs Month {n})", fontsize=14)
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Avg Active Power Lagging")
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', alpha=0.2)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_hourly_comparison_active(df, name, 12)

#### 전체 vs 1월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_active(df, name, 1)

#### 전체 vs 2월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_active(df, name, 2)

#### 전체 vs 3월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_active(df, name, 3)

#### 전체 vs 4월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_active(df, name, 4)

In [ ]:
def plot_hourly_comparison_by_month_active(df, name, n):

    # 1. 지표 설정
    metrics = {
        'activePower': 'Active Power (W)',
        'reactivePowerLagging': 'Reactive Power Lagging (Var)',
        'energy_diff': 'Energy Consumption (Wh)'
    }
    
    # 2. 그래프 생성 (3행 1열)
    fig, axes = plt.subplots(3, 1, figsize=(12, 15), sharex=True)
    
    for i, (col, ylabel) in enumerate(metrics.items()):
        # 전체 기간의 시간대별 평균
        overall_hourly = df.groupby('hour')[col].mean()
        
        # 선택한 'n월'의 시간대별 평균
        month_hourly = df[df['month'] == n].groupby('hour')[col].mean()
        
        # 시각화
        axes[i].plot(overall_hourly.index, overall_hourly.values, 
                     marker='o', color='gray', alpha=0.5, label='Entire Avg')
        axes[i].plot(month_hourly.index, month_hourly.values, 
                     marker='s', linestyle='--', color='red', label=f'Month {n} Avg')
        
        axes[i].set_ylabel(ylabel)
        axes[i].set_title(f"{name}: {col} - Overall vs Month {n}")
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

    plt.xlabel("Hour of Day (0-23)")
    plt.xticks(range(0, 24))
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_hourly_comparison_by_month(df, name, 12)

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_by_month(df, name, 1)

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_by_month(df, name, 2)

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_by_month(df, name, 3)

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_by_month(df, name, 4)

### 지연 무효전력(ReactivePowerLagging)

In [ ]:
def plot_hourly_pattern_reactive(df, name):
    
    monthly_hourly = df.pivot_table(index='hour', columns='month', values='reactivePowerLagging', aggfunc='mean')
    overall_hourly = df.groupby('hour')['reactivePowerLagging'].mean()
    
    plt.figure(figsize=(12, 6))
    
    colors = sns.color_palette("husl", len(monthly_hourly.columns))
    
    for i, month in enumerate(monthly_hourly.columns):
        plt.plot(monthly_hourly.index, monthly_hourly[month], 
                 label=f'Month {month}', color=colors[i], marker='.', alpha=0.8)
    
    plt.plot(overall_hourly.index, overall_hourly.values, 
             color='black', linewidth=2, label='OVERALL AVG', linestyle='--')
    
    plt.title(f"{name}: Hourly Reactive Power Lagging Trend by Month", fontsize=15)
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Average Reactive Power Lagging")
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', alpha=0.3)
    plt.legend(title="Months", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_hourly_pattern_reactive(df, name)

#### 전체 vs 12월

In [ ]:
def plot_hourly_comparison_reactive(df, name, n):

    # 전체 기간의 시간대별 무효전력 평균
    overall_hourly = df.groupby('hour')['reactivePowerLagging'].mean()
    # 특정 n월의 시간대별 무효전력 평균
    month_hourly = df[df['month'] == n].groupby('hour')['reactivePowerLagging'].mean()
    
    plt.figure(figsize=(12, 5))
  
    plt.plot(overall_hourly.index, overall_hourly.values, 
             color='lightgray', linewidth=3, label='Entire Avg', marker='o', alpha=0.6)
    plt.plot(month_hourly.index, month_hourly.values, 
             color='darkorange', linewidth=2, label=f'Month {n} Avg', marker='s', linestyle='--')
    plt.title(f"{name}: Hourly Reactive Power (Overall vs Month {n})", fontsize=14)
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Avg Reactive Power Lagging")
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', alpha=0.2)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 12)

#### 전체 vs 1월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 1)

#### 전체 vs 2월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 2)

#### 전체 vs 3월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 3)

#### 전체 vs 4월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 4)

### 누적 소비량(accumActiveEnergy)

In [ ]:
def plot_hourly_pattern_acc(df, name):
    
    monthly_hourly = df.pivot_table(index='hour', columns='month', values='accumActiveEnergy', aggfunc='mean')
    overall_hourly = df.groupby('hour')['accumActiveEnergy'].mean()
    
    plt.figure(figsize=(12, 6))
    
    colors = sns.color_palette("husl", len(monthly_hourly.columns))
    
    for i, month in enumerate(monthly_hourly.columns):
        plt.plot(monthly_hourly.index, monthly_hourly[month], 
                 label=f'Month {month}', color=colors[i], marker='.', alpha=0.8)
    
    plt.plot(overall_hourly.index, overall_hourly.values, 
             color='black', linewidth=2, label='OVERALL AVG', linestyle='--')
    
    plt.title(f"{name}: Hourly accumActiveEnergy Trend by Month", fontsize=15)
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Average accumActiveEnergy")
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', alpha=0.3)
    plt.legend(title="Months", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_hourly_pattern_acc(df, name)

In [ ]:
def plot_hourly_comparison_acc(df, name, n):

    # 전체 기간의 시간대별 무효전력 평균
    overall_hourly = df.groupby('hour')['accumActiveEnergy'].mean()
    # 특정 n월의 시간대별 무효전력 평균
    month_hourly = df[df['month'] == n].groupby('hour')['accumActiveEnergy'].mean()
    
    plt.figure(figsize=(12, 5))
  
    plt.plot(overall_hourly.index, overall_hourly.values, 
             color='lightgray', linewidth=3, label='Entire Avg', marker='o', alpha=0.6)
    plt.plot(month_hourly.index, month_hourly.values, 
             color='darkorange', linewidth=2, label=f'Month {n} Avg', marker='s', linestyle='--')
    plt.title(f"{name}: Hourly accumActiveEnergy (Overall vs Month {n})", fontsize=14)
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Avg accumActiveEnergy")
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', alpha=0.2)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_hourly_comparison_acc(df, name, 12)

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 1)

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 2)

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 3)

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_reactive(df, name, 4)

### 에너지 소비량(energy_diff)

In [ ]:
def plot_hourly_pattern_diff(df, name):
    
    monthly_hourly = df.pivot_table(index='hour', columns='month', values='energy_diff', aggfunc='mean')
    overall_hourly = df.groupby('hour')['energy_diff'].mean()
    
    plt.figure(figsize=(12, 6))
    
    colors = sns.color_palette("husl", len(monthly_hourly.columns))
    
    for i, month in enumerate(monthly_hourly.columns):
        plt.plot(monthly_hourly.index, monthly_hourly[month], 
                 label=f'Month {month}', color=colors[i], marker='.', alpha=0.8)
    
    plt.plot(overall_hourly.index, overall_hourly.values, 
             color='black', linewidth=2, label='OVERALL AVG', linestyle='--')
    
    plt.title(f"{name}: Hourly energy_diff Trend by Month", fontsize=15)
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Average energy_diff")
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', alpha=0.3)
    plt.legend(title="Months", bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_hourly_pattern_diff(df, name)

#### 전체 vs 12월

In [ ]:
def plot_hourly_comparison_diff(df, name, n):

    # 전체 기간의 시간대별 무효전력 평균
    overall_hourly = df.groupby('hour')['energy_diff'].mean()
    # 특정 n월의 시간대별 무효전력 평균
    month_hourly = df[df['month'] == n].groupby('hour')['energy_diff'].mean()
    
    plt.figure(figsize=(12, 5))
  
    plt.plot(overall_hourly.index, overall_hourly.values, 
             color='lightgray', linewidth=3, label='Entire Avg', marker='o', alpha=0.6)
    plt.plot(month_hourly.index, month_hourly.values, 
             color='darkorange', linewidth=2, label=f'Month {n} Avg', marker='s', linestyle='--')
    plt.title(f"{name}: Hourly energy_diff (Overall vs Month {n})", fontsize=14)
    plt.xlabel("Hour of Day (0-23)")
    plt.ylabel("Avg energy_diff")
    plt.xticks(range(0, 24))
    plt.grid(True, axis='y', alpha=0.2)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

for df, name in zip(df_list, names):
    plot_hourly_comparison_diff(df, name, 12)

#### 전체 vs 1월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_diff(df, name, 1)

#### 전체 vs 2월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_diff(df, name, 2)

#### 전체 vs 3월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_diff(df, name, 3)

#### 전체 vs 4월

In [ ]:
for df, name in zip(df_list, names):
    plot_hourly_comparison_diff(df, name, 4)

## 유효전력 vs 지연 무효전력 패턴

In [ ]:
df2['powerFactorR'] + df2['powerFactorS'] + df2['powerFactorT']

In [ ]:
df2.columns

In [ ]:
import seaborn as sns

sns.pairplot(df2)